In [1]:
%pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amontgomerie/cefr-levelled-english-texts")

print("Path to dataset files:", path)

Path to dataset files: /Users/theidol/.cache/kagglehub/datasets/amontgomerie/cefr-levelled-english-texts/versions/1


In [3]:
import pandas as pd

In [4]:
texts = pd.read_csv(f"{path}/cefr_leveled_texts.csv")

In [5]:
texts.head()

,text,label
0,Hi!\nI've been meaning to write for ages and f...,B2
1,﻿It was not so much how hard people found the ...,B2
2,Keith recently came back from a trip to Chicag...,B2
3,"The Griffith Observatory is a planetarium, and...",B2
4,-LRB- The Hollywood Reporter -RRB- It's offici...,B2


In [6]:
texts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1494 entries, 0 to 1493
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1494 non-null   object
 1   label   1494 non-null   object
dtypes: object(2)
memory usage: 23.5+ KB


In [7]:
label_counts = texts.groupby("label").size().reset_index(name="count")

label_counts["percentage"] = (
    label_counts["count"] / label_counts["count"].sum() * 100
).round(2)


print(label_counts)

  label  count  percentage
0    A1    288       19.28
1    A2    272       18.21
2    B1    205       13.72
3    B2    286       19.14
4    C1    241       16.13
5    C2    202       13.52


In [8]:
texts.to_csv("cefr_leveled_texts.csv", index=False)

In [9]:
texts["word_count"] = texts["text"].str.split().str.len()
texts["num_sentences"] = texts["text"].str.count(r"[.!?]")


In [10]:
texts["num_sentences"] = texts["num_sentences"].replace(0, 1)

In [11]:
texts["avg_sentence_length"] = texts["word_count"] / texts["num_sentences"]

In [12]:
texts.head()

,text,label,word_count,num_sentences,avg_sentence_length
0,Hi!\nI've been meaning to write for ages and f...,B2,459,27,17.000000
1,﻿It was not so much how hard people found the ...,B2,677,34,19.911765
2,Keith recently came back from a trip to Chicag...,B2,236,14,16.857143
3,"The Griffith Observatory is a planetarium, and...",B2,301,17,17.705882
4,-LRB- The Hollywood Reporter -RRB- It's offici...,B2,352,20,17.600000


In [13]:


label_stats = (
    texts
    .groupby("label")
    .agg(
        avg_word_count=("word_count", "mean"),
        min_word_count=("word_count", "min"),
        max_word_count=("word_count", "max"),
        avg_num_sentences=("num_sentences", "mean"),
        avg_sentence_length=("avg_sentence_length", "mean"),
        sample_count=("text", "count")
    )
    .round(2)
)

cefr_order = ["A1", "A2", "B1", "B2", "C1", "C2"]

label_stats = label_stats.reindex(cefr_order)

In [14]:
print(label_stats)

       avg_word_count  min_word_count  max_word_count  avg_num_sentences  \
label                                                                      
A1              95.00              34             298              16.20   
A2             227.72              65            1216              22.60   
B1             409.56             100            1630              28.79   
B2             498.58              97            1666              29.19   
C1             683.66             153            1662              34.83   
C2             688.98             114            2227              30.35   

       avg_sentence_length  sample_count  
label                                     
A1                    6.17           288  
A2                    9.85           272  
B1                   14.56           205  
B2                   17.13           286  
C1                   20.10           241  
C2                   23.33           202  


In [15]:
texts.to_csv("cefr_level_texts_with stats.csv", index=False)